# Proyecto final — Inventario TI y diagnóstico de servicios

Este notebook reproduce el flujo del proyecto paso a paso. Está pensado para revisar la solución sin usar `input()` y sin bloquear la ejecución.


## 1. Preparar rutas

Usamos `pathlib.Path` para trabajar con rutas de forma portable.


In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "datos"
SALIDA_DIR = BASE_DIR / "salida"
LOG_DIR = BASE_DIR / "logs"

print("Proyecto:", BASE_DIR)
print("Datos:", DATA_DIR)


## 2. Leer la política JSON

La política contiene umbrales, estados válidos y puertos esperados por servicio.


In [ ]:
from inventario_ti.io_datos import cargar_politica

politica = cargar_politica(DATA_DIR / "politica_servicios.json")
politica


## 3. Leer el inventario CSV

El módulo `io_datos` convierte cada fila en un objeto `Equipo` y transforma los campos numéricos.


In [ ]:
from inventario_ti.io_datos import cargar_equipos

equipos = cargar_equipos(DATA_DIR / "equipos.csv")
print("Equipos cargados:", len(equipos))
print(equipos[0])


## 4. Revisar objetos y colecciones

Cada elemento de la lista es una instancia de la clase `Equipo`. Podemos obtener servicios únicos con un conjunto.


In [ ]:
servicios = {equipo.servicio for equipo in equipos}
entornos = sorted({equipo.entorno for equipo in equipos})

print("Servicios:", sorted(servicios))
print("Entornos:", entornos)


## 5. Validar y clasificar

El análisis combina funciones, métodos de los objetos, diccionarios y reglas de la política.


In [ ]:
from inventario_ti.analisis import enriquecer_equipos, obtener_resumen, filtrar_pendientes

equipos_enriquecidos = enriquecer_equipos(equipos, politica)
resumen = obtener_resumen(equipos_enriquecidos)
pendientes = filtrar_pendientes(equipos_enriquecidos)

print("Resumen:")
print(resumen)
print()
print("Pendientes:", len(pendientes))


## 6. Mostrar equipos pendientes

Esta vista ayuda al técnico de soporte a localizar los equipos que requieren atención.


In [ ]:
for equipo in pendientes:
    print(f"{equipo['nombre']:15} {equipo['entorno']:8} {equipo['servicio']:12} {equipo['clasificacion']:8} {equipo['incidencias']}")


## 7. Generar salidas

Se generan los mismos ficheros que produciría `main.py`: informe de texto y JSON de resumen.


In [ ]:
from inventario_ti.reportes import generar_informe
from inventario_ti.io_datos import guardar_json, guardar_texto

informe = generar_informe(resumen, pendientes)
guardar_texto(SALIDA_DIR / "informe.txt", informe)
guardar_json(SALIDA_DIR / "resumen.json", resumen)
guardar_json(SALIDA_DIR / "equipos_clasificados.json", equipos_enriquecidos)

print(informe)


## 8. Ejecutar el flujo completo desde código

También podemos invocar la función `ejecutar()` de `main.py` para verificar la integración completa.


In [ ]:
from main import configurar_logging, ejecutar

configurar_logging()
resumen_final, pendientes_final = ejecutar(DATA_DIR / "equipos.csv", DATA_DIR / "politica_servicios.json")
print("Total:", resumen_final["total_equipos"])
print("Pendientes:", len(pendientes_final))
